# 🚀 Step 2: Ollama + cloudflared 公開APIサーバー

**orcarouter/Qwen3.8-27B-Uncensored:latest** を Kaggle の T4 x2 で動かし、cloudflared で公開します。

---

### 前提条件（これだけ）

1. **Accelerator**: `GPU T4 x2` に設定
2. **Internet**: `On` に設定
3. **Add Data**: 以下の **3つ** を追加
   - `my-ollama-orcarouter-qwen38-27b`（モデル 16.5 GiB）
   - `my-ollama-orcarouter-qwen38-27b-patch`（ollama + llama-server + CUDA 1.45 GB）
   - `my-cloudflared-bin`（cloudflared 40 MB）— 無いと毎回 GitHub から落とす
> ✅ **Secrets の設定は不要です。**
> cloudflared の quick tunnel はアカウント不要。authtoken も API キーも要りません。
> 自動で発行される公開URLがそのままエンドポイントになります。
>
> 🔓 **認証はありません**（URL を知れば誰でも使えます）。
> 他人に使われたくないときは環境変数 `API_PASS` を設定してください
> （proxy が Basic 認証を有効化します）。
>
> ⚠ 公開URLは **実行のたびに変わります**（quick tunnel の仕様）。
> 切れたらトンネルセルを再実行し、接続情報を取り直してください。
> 現在の URL はビーコン（`python3 -m cli beacon`）から読めます。

---

### このモデルについて

| 項目 | 値 |
|---|---|
| 量子化 | Q4_K_M |
| パラメータ | 27.3B |
| サイズ | 16.52 GiB |
| コンテキスト | 262,144（このノートブックでは 131,072 で運用） |
| 機能 | completion / vision / tools / thinking |

Abliterlitics の Qwen3.8-27B 12変体比較で総合1位（judge ASR 82.2%、capability 劣化なし）。
手元の `Qwen3.8-27B-Uncensored-Q4_K_M.gguf` とは sha256 完全一致。


---
### 🧩 共通ランタイム（proxy / tunnel / keepalive / verify）

`runtime/` の真実源をビルダーが連結して埋め込んでいます。
セルを直接編集しても次のビルドで上書きされます。


In [ ]:
# ============================================================
# 共通ランタイム（runtime/*.py から自動生成。手で編集しない）
# proxy / tunnel / keepalive / verify の配管をここに集約する。
# ============================================================
# ── runtime/net.py ────────────────────────────────────────
"""トンネル URL 取得と HTTP 疎通の共通処理（生成コードへ埋め込む）。"""
import json
import re
import socket
import time
import urllib.error
import urllib.request

# hosts は表駆動。今はログ解析の 1 種類しか無いが、待ち時間を URL の出所と
# 一緒に置いておくと `kc_wait_url` が既定値を知らずに済む。表に無い名前は
# KeyError になる（黙って空 URL を返さない）。
TUNNEL_HOSTS = {
    "cloudflared": {
        "url_re": re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com"),
        "wait_seconds": 90,
    },
}


# Dataset 同梱バイナリと同じ版に固定する（latest は再現性がない）。
KC_CLOUDFLARED_VERSION = "2026.10.4"


def kc_cloudflared_url(version=KC_CLOUDFLARED_VERSION):
    """cloudflared (linux amd64) のダウンロード URL。"""
    return ("https://github.com/cloudflare/cloudflared/releases/download/"
            f"{version}/cloudflared-linux-amd64")


def kc_probe_url(host, log_path=None):
    """現在の公開URLを返す（無ければ ""）。host は cloudflared のみ。"""
    spec = TUNNEL_HOSTS[host]
    if not log_path:
        return ""
    try:
        with open(log_path) as f:
            m = spec["url_re"].search(f.read())
        return m.group(0) if m else ""
    except OSError:
        return ""


def kc_wait_url(host, log_path=None, seconds=None, interval=1, progress_every=10):
    """公開URLが出るまで待つ。seconds 未指定なら host ごとの既定値。"""
    if seconds is None:
        seconds = TUNNEL_HOSTS[host]["wait_seconds"]
    for i in range(seconds):
        url = kc_probe_url(host, log_path)
        if url:
            return url
        if progress_every and i % progress_every == progress_every - 1:
            print(f"⏳ URL 待ち ({i + 1}/{seconds})...")
        time.sleep(interval)
    return ""


def kc_urlopen_v4(url, timeout=15):
    """公開 URL を開く。失敗したら IPv4 優先で名前解決して 1 回だけ再試行。

    Kaggle の VM は AAAA（IPv6）を返すのに IPv6 経路が無いことがあり、
    `urlopen` が "Network is unreachable" で落ちる（dryrun で実測）。
    自分の URL に届かないだけでトンネルは生きているので、ここで諦めると
    keepalive が**生きているトンネルを再起動して URL が変わってしまう**。
    """
    try:
        return urllib.request.urlopen(url, timeout=timeout)
    except Exception as first:
        err = first
    orig = socket.getaddrinfo

    def _v4(host, port, family=0, type=0, proto=0, flags=0):
        return orig(host, port, socket.AF_INET, type, proto, flags)

    socket.getaddrinfo = _v4
    try:
        return urllib.request.urlopen(url, timeout=timeout)
    except Exception:
        raise err
    finally:
        socket.getaddrinfo = orig


def kc_wait_http(url, attempts=20, interval=3, timeout=10, opener=None):
    """200 が返るまで待つ。成功したら経過秒（(i+1)*interval）、失敗なら None。"""
    opener = opener or urllib.request.urlopen
    for i in range(attempts):
        try:
            with opener(url, timeout=timeout) as r:
                if r.status == 200:
                    return (i + 1) * interval
        except Exception:
            pass
        time.sleep(interval)
    return None


def kc_retry(fn, tries=6, delay=5, skip_http_errors=True):
    """一過性の通信エラーを吸収する。HTTP 応答は「答え」なので既定で再試行しない。"""
    last = None
    for i in range(max(1, tries)):
        try:
            return fn()
        except urllib.error.HTTPError as e:
            if skip_http_errors:
                raise
            last = e
        except Exception as e:
            last = e
        if i + 1 < tries:
            wait = delay * (i + 1)
            print(f"  ⚠ 一時的な通信エラー（{i + 1}/{tries}）: {last} "
                  f"→ {wait}s 待って再試行")
            time.sleep(wait)
    raise last


def kc_post_json(url, payload, timeout=60, headers=None, opener=None):
    """JSON POST のレスポンスを返す（ストリームは呼び出し側で読む）。"""
    h = {"Content-Type": "application/json"}
    if headers:
        h.update(headers)
    req = urllib.request.Request(
        url, data=json.dumps(payload).encode(), headers=h, method="POST")
    return (opener or urllib.request.urlopen)(req, timeout=timeout)

# ── runtime/proc.py ───────────────────────────────────────
"""プロセス・GPU 状態の確認と後片付け（生成コードへ埋め込む）。"""
import re
import subprocess
import time


def _kc_live(pids, timeout=5):
    """zombie を除いた PID を返す。kill 直後は未 reap の Z が pgrep に残る。"""
    if not pids:
        return []
    try:
        out = subprocess.run(
            ["ps", "-o", "pid=,stat=", "-p", ",".join(pids)],
            capture_output=True, text=True, timeout=timeout,
        ).stdout
    except Exception:
        return pids
    live = []
    for line in out.splitlines():
        parts = line.split()
        if len(parts) >= 2 and not parts[1].startswith("Z"):
            live.append(parts[0])
    return live


def kc_proc_state(name, match="exact", timeout=5):
    """alive / DEAD / unknown。match=exact は pgrep -x、argv は pgrep -f。"""
    try:
        flag = "-x" if match == "exact" else "-f"
        out = subprocess.run(["pgrep", flag, name], capture_output=True,
                             text=True, timeout=timeout).stdout
        pids = [p for p in out.split() if p.strip()]
        return "alive" if _kc_live(pids, timeout) else "DEAD"
    except Exception:
        return "unknown"


def kc_gpu_usage(timeout=5):
    """nvidia-smi の使用率/VRAM を 1 行で返す（CPU 環境では N/A）。"""
    try:
        return subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=timeout,
        ).stdout.strip().replace("\n", " | ")
    except Exception:
        return "N/A"


def kc_vram_used(timeout=5):
    """GPU ごとの使用 VRAM (MiB)。取れなければ []。"""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=timeout,
        ).stdout
        return [int(x) for x in out.split() if x.strip().isdigit()]
    except Exception:
        return []


def kc_sample(pid, root="/proc"):
    """1 プロセスの様子を 1 つの dict にまとめる。取れない項目は None。

    `read_mib` は **実際にブロックデバイスまで行った読み込み量**（/proc の
    read_bytes）。ページキャッシュに当たった分は含まれないので、
    「重みを読み終えたか」はこちらで見る。`char_mib` は read(2) の総量で、
    キャッシュ命中も含む。両方出すのは、この 2 つが離れたときに
    「キャッシュから読んでいる」と言い切れるようにするため。

    `root` はテスト用の口。実プロセスの RSS は環境で変わるので、そこを
    assert すると「たまたま通る」テストになる。`/proc` の形をした
    ディレクトリを渡せば、変換（kB → MiB など）だけを固定できる。
    """
    s = {"rss_mib": None, "read_mib": None, "char_mib": None, "vram_mib": None}
    try:
        with open(f"{root}/{pid}/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    s["rss_mib"] = round(int(line.split()[1]) / 1024)
                    break
    except OSError:
        pass
    try:
        with open(f"{root}/{pid}/io") as f:
            for line in f:
                if line.startswith("read_bytes:"):
                    s["read_mib"] = int(line.split()[1]) // 2**20
                elif line.startswith("rchar:"):
                    s["char_mib"] = int(line.split()[1]) // 2**20
    except OSError:
        pass
    vram = kc_vram_used()
    if vram:
        s["vram_mib"] = sum(vram)
    return s


def kc_watch(pid, every=30, keep=80):
    """待機中のプロセスを `every` 秒ごとに観測する。戻り値は `(samples, stop)`。

    `stop()` を呼ぶと観測を止める。**プロセスの死だけでは止まらない**設計に
    してあるのは、待っている相手がサーバー本体だから: 起動が通れば同じ
    PID が何時間も生き続けるので、死を待つと `[watch]` が 30 秒ごとに
    何百行も出てログを埋める。呼び出し側は待機が終わった時点で止めること。

    無言の待機を「遅い」と「固まった」に分けるための計器。実測 2026-11-01
    exl3: 1800 秒の待機中に chat_server.log が **0 バイト**で、読み込みが
    進んでいたのか止まっていたのかまったく分からなかった（エンジン側の
    進捗表示は `--no-progress` で切ってあり、完走時の 1 行しか出ない）。
    そうなると次の手は「もう 1 回回す」しかなく、1 往復 30 分を捨てる。

    観測は read_bytes の**増分**が主役。1800 秒で 3 GiB しか進んでいなければ
    1.7 MiB/s ということなので、遅いのか止まったのかは即座に判る。

    戻り値の samples はそのまま result.json に入れる。失敗時にも
    「どこまで進んでいたか」が残るのが目的なので、成功・失敗を問わず入れる。
    """
    import os
    import threading

    samples = []
    stop = threading.Event()

    def loop():
        t0 = time.time()
        while not stop.wait(every):        # 止める指示が来たら即座に抜ける
            s = kc_sample(pid)
            s["t"] = round(time.time() - t0)
            samples.append(s)
            del samples[:-keep]
            print(f"  [watch] {s['t']}s rss={s['rss_mib']}MiB "
                  f"read={s['read_mib']}MiB vram={s['vram_mib']}MiB",
                  flush=True)
            try:
                os.kill(pid, 0)      # 死んでいれば ProcessLookupError
            except OSError:
                return

    threading.Thread(target=loop, daemon=True).start()
    return samples, stop.set


def kc_context_budget(vram_total_mib, model_gib, kv_kib_per_token, candidates,
                      requested=None, budget_ratio=0.88):
    """VRAM から (コンテキスト長, 予算 GiB) を決める。

    予算は実効 VRAM の `budget_ratio` 倍（T4 x2 の 30 GiB をそのまま
    使うと CUDA graph / activation で溢れるため 0.88 で見積もる）。
    `requested` が収まるならそれを優先し、駄目なら `candidates` を
    大きい順に試す。どれも収まらなければ最小候補を返す（起動は試す）。
    """
    budget_gib = vram_total_mib / 1024 * budget_ratio

    def fits(ctx):
        return model_gib + kv_kib_per_token * ctx / 1024 / 1024 <= budget_gib

    if requested and fits(requested):
        return requested, budget_gib
    for ctx in candidates:
        if fits(ctx):
            return ctx, budget_gib
    return candidates[-1], budget_gib


def kc_context_gib(model_gib, kv_kib_per_token, ctx):
    """コンテキスト 1 本ぶんの見積り（モデル + KV）を GiB で返す。"""
    return model_gib + kv_kib_per_token * ctx / 1024 / 1024


def kc_detect_gpu(timeout=15):
    """全 GPU の (名前, VRAM MiB) を返す。検出失敗は ([], [])。"""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=timeout,
        ).stdout.strip()
        names, mems = [], []
        for line in out.splitlines():
            if not line.strip():
                continue
            parts = line.split(",")
            names.append(parts[0].strip())
            mems.append(int(re.sub(r"[^0-9]", "", parts[1])))
        return names, mems
    except Exception as e:
        print(f"GPU 検出失敗: {e}")
        return [], []


def kc_wait_gpu_idle(limit_mib=500, timeout=180):
    """全 GPU の使用量が limit 未満になるまで待つ。経過秒、失敗は -1。

    名前は ollama の kc_wait_vram_free（/api/ps を見る）と衝突させない。
    """
    for i in range(timeout):
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
        used = [int(x) for x in out.splitlines() if x.strip().isdigit()]
        if used and max(used) < limit_mib:
            return i
        time.sleep(1)
    return -1


def _kc_argv(pattern):
    """"-x cloudflared" → ["-x", "cloudflared"]（shell を介さない）。"""
    return pattern.split()


def kc_pids(pattern, timeout=15):
    """pgrep の引数文字列（例: "-x cloudflared"）に一致する PID を返す。"""
    try:
        r = subprocess.run(["pgrep", *_kc_argv(pattern)],
                           capture_output=True, text=True, timeout=timeout)
        return _kc_live([p for p in r.stdout.split() if p.strip()], timeout)
    except Exception:
        return []


def kc_stop_processes(targets, term_wait=3, final_wait=2):
    """(pgrep_pattern, label) を TERM → 待機 → KILL で止め、残存ラベルを返す。"""
    for pattern, label in targets:
        if kc_pids(pattern):
            subprocess.run(["pkill", "-TERM", *_kc_argv(pattern)],
                           capture_output=True)
            print(f"  TERM → {label}")
    time.sleep(term_wait)
    for pattern, label in targets:
        if kc_pids(pattern):
            subprocess.run(["pkill", "-KILL", *_kc_argv(pattern)],
                           capture_output=True)
            print(f"  KILL → {label}（残存していた）")
    time.sleep(final_wait)
    return [label for pattern, label in targets if kc_pids(pattern)]

# ── runtime/storage.py ────────────────────────────────────
"""検証成果物の永続化と失敗記録（生成コードへ埋め込む）。"""
import glob
import os

# `kaggle kernels output` が zip 化するのは /kaggle/working だけ。
# /output は公式の永続先ではないが、書ければ保険になるので両方へ書く。
KC_OUT_DIRS = ("/kaggle/working", "/output")


def kc_find_dataset(slug, root="/kaggle/input"):
    """Dataset のマウント先を返す（見つからなければ既定パス）。

    Kaggle は `/kaggle/input/<slug>`（旧）と
    `/kaggle/input/datasets/<owner>/<slug>`（新）が同一日内に混在する。
    深さ 2 まで探し、**空でない**ディレクトリを優先する（空マウントを
    掴むと「Dataset はあるのに中身が無い」で落ちるため）。
    呼び出し側は戻り値をそのまま使ってよい（従来の `_find_input` と同じ）。
    """
    cands = []
    for pat in (f"{root}/{slug}", f"{root}/*/{slug}", f"{root}/*/*/{slug}"):
        cands += [p for p in glob.glob(pat) if os.path.isdir(p)]
    for p in sorted(cands):
        try:
            if os.listdir(p):
                return p
        except OSError:
            continue
    return cands[0] if cands else f"{root}/{slug}"


def kc_save_output(name, text, dirs=KC_OUT_DIRS):
    """1 つでも書ければ True（成功条件に使う）。"""
    saved = []
    for d in dirs:
        try:
            os.makedirs(d, exist_ok=True)
            with open(os.path.join(d, name), "w") as f:
                f.write(text)
            saved.append(d)
        except Exception as e:
            print(f"  ⚠ {d} 書き出し失敗 ({e})")
    if saved:
        print(f"  📄 {name} 保存済: {', '.join(saved)}")
        return True
    print(f"  ⚠ 成果物を1つも保存できませんでした: {name}")
    return False


def kc_fail(msg, tail_path=None, dirs=KC_OUT_DIRS):
    """失敗原因を永続領域に残してから raise（ログ API は当てにできない）。"""
    tail = ""
    try:
        if tail_path and os.path.isfile(tail_path):
            tail = open(tail_path, errors="replace").read()[-3000:]
    except Exception:
        pass
    kc_save_output("FAILED.txt", f"{msg}\n\n{tail}", dirs)
    raise RuntimeError(msg)

# ── runtime/keepalive.py ──────────────────────────────────
"""キープアライブ設定ファイルと /stop の共通処理（生成コードへ埋め込む）。"""
import json
import os
import urllib.request


def kc_read_conf(path, default_minutes=60, max_minutes=480,
                 on_invalid="fail", fail=None, quiet_clamp=False):
    """(run, minutes, mtime) を返す。RUN=0 は (0, default_minutes, mtime)。

    on_invalid: "fail" は fail(msg)（未指定なら ValueError）、
                "zero" は停止扱い（RUN=0 と同じ）。
    quiet_clamp: proxy が書く既定値を上限で頭打ちにするとき黙る。
    """
    try:
        with open(path) as f:
            d = dict(l.strip().split("=", 1) for l in f
                     if l.strip() and "=" in l)
        run = int(d.get("RUN", 1))
        minutes = int(d.get("MINUTES", default_minutes))
    except OSError:
        run, minutes = 1, default_minutes
    except ValueError:
        run, minutes = 1, default_minutes
    try:
        mtime = os.path.getmtime(path)
    except OSError:
        mtime = None
    if run <= 0:
        return 0, default_minutes, mtime
    if minutes <= 0:
        msg = f"KEEPALIVE_MINUTES={minutes} は無効（1〜{max_minutes} で指定）"
        if on_invalid == "zero":
            return 0, default_minutes, mtime
        if fail:
            fail(msg)
        raise ValueError(msg)
    if minutes > max_minutes:
        if not (quiet_clamp and minutes == default_minutes):
            print(f"⚠ KEEPALIVE_MINUTES={minutes} → {max_minutes} に制限")
        minutes = max_minutes
    return run, minutes, mtime


def kc_write_conf(path, run, minutes=None):
    with open(path, "w") as f:
        f.write(f"RUN={run}\n")
        if minutes is not None:
            f.write(f"MINUTES={minutes}\n")


def kc_conf_mtime(path):
    """keepalive.conf の更新時刻。proxy がチャット送信ごとに書き直す。"""
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0.0


def kc_keepalive_deadline(started, minutes, mtime, hard_max_minutes=180):
    """停止時刻。最後の活動（起動 or conf 更新）から minutes 分、全体は hard_max で頭打ち。"""
    last = max(started, mtime or started)
    return min(last + minutes * 60, started + hard_max_minutes * 60)


def kc_keepalive_expired(now, started, minutes, mtime, hard_max_minutes=180):
    """now 時点で時間予算を超えたか（now を渡すのでフェイククロックで検証できる）。"""
    return now >= kc_keepalive_deadline(started, minutes, mtime,
                                        hard_max_minutes)


def kc_stop_via_proxy(port, auth=None, timeout=30):
    headers = {"X-Kc-Stop": "1"}
    if auth:
        headers["Authorization"] = auth
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/stop", data=b"", method="POST",
        headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, json.loads(r.read())

# ── runtime/proxy.py ──────────────────────────────────────
"""auth_proxy の起動と認証なし検査（生成コードへ埋め込む）。"""
import subprocess
import time
import urllib.error
import urllib.request


def kc_start_proxy(port, upstream, chat_html, proxy_path,
                   extra_args=(), env=None, log_path="/tmp/auth_proxy.log",
                   wait_seconds=30, label="プロキシ", fail=None,
                   python="python3"):
    """auth_proxy.py を起動し /healthz を待つ。失敗時は fail() か RuntimeError。"""
    subprocess.run(["pkill", "-f", "auth_proxy.py"], capture_output=True)
    time.sleep(1)
    cmd = [python, proxy_path, "--port", str(port),
           "--upstream", upstream, "--chat-html", chat_html, *extra_args]
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=log)
    print(f"🔄 proxy PID: {proc.pid}")

    elapsed = None
    for i in range(wait_seconds):
        try:
            with urllib.request.urlopen(
                    f"http://127.0.0.1:{port}/healthz", timeout=3) as r:
                if r.status == 200:
                    elapsed = i + 1
                    break
        except Exception:
            pass
        time.sleep(1)
    if elapsed is None:
        try:
            print(open(log_path).read()[-2000:])
        except OSError:
            pass
        msg = f"{label}の起動に失敗しました（{log_path}）"
        if fail:
            fail(msg, log_path)
        raise RuntimeError(msg)
    print(f"✅ {label}起動完了（{elapsed}s）")
    return proc


def kc_assert_no_auth(port, path="/api/tags", fail=None, timeout=10):
    """401/403 なら認証が有効 = 設定ミス。上流未起動の 5xx は正常。"""
    req = urllib.request.Request(f"http://127.0.0.1:{port}{path}")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            print(f"  ✅ 認証なし → HTTP {r.status}")
            return True
    except urllib.error.HTTPError as e:
        if e.code in (401, 403):
            msg = f"認証なしで HTTP {e.code} — 認証が有効になっています"
            if fail:
                fail(msg)
            raise RuntimeError(msg)
        print(f"  ✅ 認証なし → HTTP {e.code}（401 でない）")
        return True
    except Exception as e:
        msg = f"プロキシ検査に失敗: {type(e).__name__}: {e}"
        if fail:
            fail(msg)
        raise

# ── runtime/engines/ollama.py ─────────────────────────────
"""Ollama 固有の推論・モデル解放処理（生成コードへ埋め込む）。"""
import json
import time
import urllib.request


def kc_ollama_chat(port, model, prompt, num_predict=64, timeout=1800,
                   think=None):
    """Ollama /api/chat を 1 回叩き、応答 dict を返す（非ストリーム）。

    think=None はボディにキーを入れない（Ollama 既定に任せる）。
    """
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"num_predict": num_predict, "temperature": 0},
    }
    if think is not None:
        body["think"] = think
    req = urllib.request.Request(
        f"http://127.0.0.1:{port}/api/chat",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


def kc_unload_model(port, model, timeout=60):
    """keep_alive=0 でモデルを VRAM から降ろす。1 経路でも通れば True。

    プロセスを殺すより先に、Ollama 自身に VRAM を返させる。
    """
    unloaded = False
    for path, payload in [
        ("/api/generate", {"model": model, "keep_alive": 0}),
        ("/api/chat", {"model": model, "messages": [], "keep_alive": 0}),
    ]:
        try:
            req = urllib.request.Request(
                f"http://127.0.0.1:{port}{path}",
                data=json.dumps(payload).encode(),
                headers={"Content-Type": "application/json"}, method="POST")
            with urllib.request.urlopen(req, timeout=timeout) as r:
                if r.status == 200:
                    unloaded = True
                    print(f"  ✅ アンロード要求 OK ({path})")
                    break
        except Exception as e:
            print(f"  —  {path}: {type(e).__name__}")
    return unloaded


def kc_wait_vram_free(port, attempts=10, interval=3):
    """常駐モデルが 0 になるまで待つ。空なら True。"""
    for attempt in range(attempts):
        try:
            with urllib.request.urlopen(
                    f"http://127.0.0.1:{port}/api/ps", timeout=10) as r:
                models = json.loads(r.read())
            loaded = models.get("models") or []
            if not loaded:
                print("  ✅ VRAM 解放確認（常駐モデル 0）")
                return True
            vram = sum(m.get("size_vram", 0) for m in loaded)
            print(f"  ⏳ まだ常駐: {vram:,} B（{attempt + 1}/{attempts}）")
        except Exception as e:
            print(f"  —  /api/ps 確認失敗: {type(e).__name__}")
            return False
        time.sleep(interval)
    print("  ⚠ アンロードを確認できず（プロセス kill へ進む）")
    return False

# ── runtime/beacon.py ─────────────────────────────────────
"""ビーコンカーネル（Kaggle のカーネルを掲示板として使う）。

**カーネルのコードそのもの**を公開 URL の置き場にする。

- 書き手: 配信カーネルの中から `kernels_push` で小さなカーネルを更新する。
  認証は Kaggle がカーネルへ注入するもの（Secrets 不要）。
- 読み手: `kaggle kernels pull` でコードを引いて `URL` を読む。

外部サービスに URL を置く方式をやめた理由: Kaggle の Secret は**カーネル
ごとに紐付ける**必要があり、未紐付けの `get_secret()` は `ConnectionError`
を返す（通信障害に見える）。専用アカウント運用では、カーネルを増やすたびに
UI 作業が要り、トークンも個人アカウントのものが残る。ビーコンは
**アカウント・Secrets・外部サービスを増やさずに**同じことができる。
経緯は `notes/url-publishing.md`。

この層の方針: **失敗しても配信を止めない。ただし理由は必ず返す。**
外部の置き場が 3 日間更新されていないことに誰も気づかない、という失敗を
しているので、`(ok, detail)` を返さない関数はここに置かない。

`kc_` 接頭辞は生成コードの名前空間で衝突しないため（runtime/__init__.py）。
"""
import json
import os
import shutil
import sys
import tempfile
import time

# 掲示板の中身。**この文字列がそのまま beacon カーネルの main.py になる。**
# 値は repr で埋めるので、URL に引用符が混ざっても壊れない。
_BEACON_BODY = '''"""公開 URL のビーコン（掲示板）。

このカーネルは**実行しない**。コードそのものが受け渡しの場所で、
配信カーネルが `kernels_push` でこのファイルを書き換える。
読む側は `kaggle kernels pull` して `URL` と `UPDATED` を読む。

`STAGE` は `serving` / `probe` など。**`probe` は本物ではない**ので、
読む側は `probe` を捨てること（テスト URL を掴む事故を防ぐ）。
`UPDATED` は発行時刻の epoch 秒。古い値を掴んでいないかの判定に使う。
"""

URL = {url!r}
STAGE = {stage!r}
UPDATED = {updated!r}
'''


def kc_beacon_slug(slug=""):
    """`owner/name` を解決する（引数 → 環境変数）。無ければ空文字。"""
    s = (slug or "").strip()
    if s:
        return s
    return (os.environ.get("KC_BEACON_SLUG", "") or "").strip()


def kc_kaggle_path(entries=None):
    """`import kaggle` を壊すパスを取り除いた `sys.path` を返す。

    リポジトリ直下には `kaggle/`（カーネルのビルド元）がある。これが
    **インストール済みの `kaggle` パッケージを隠す**ので、ローカルで
    `cli beacon` を叩くと `No module named 'kaggle.api'` で死ぬ（実測）。
    カーネル内には `kaggle/` ディレクトリが無いので、ここは素通りする。

    判定は「`kaggle/` があり、かつ `kaggle/api` が無い」パスだけを外す。
    `kaggle/api` を持つ本物の候補は残すので、site-packages は消えない。
    """
    src = list(sys.path if entries is None else entries)
    keep = []
    for p in src:
        base = p or "."
        try:
            shadow = (os.path.isdir(os.path.join(base, "kaggle"))
                      and not os.path.isdir(os.path.join(base, "kaggle", "api")))
        except OSError:                       # 権限の無いパスは触らない
            shadow = False
        if not shadow:
            keep.append(p)
    return keep


def kc_kaggle_api():
    """`KaggleApi` クラスを返す。無ければ例外を投げる（呼び出し側が拾う）。

    普通に import して駄目だったときだけ `kc_kaggle_path` で探し直す。
    一度失敗した `kaggle` は名前空間パッケージとして `sys.modules` に
    残っているので、それも捨てないと再試行が同じ場所へ行く。
    """
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        return KaggleApi
    except Exception:
        saved = list(sys.path)
        try:
            sys.path[:] = kc_kaggle_path(saved)
            sys.modules.pop("kaggle", None)
            sys.modules.pop("kaggle.api", None)
            from kaggle.api.kaggle_api_extended import KaggleApi
            return KaggleApi
        finally:
            sys.path[:] = saved


def kc_beacon_meta(slug):
    """beacon カーネルの kernel-metadata.json を組み立てる。

    **実行しない掲示板**なので GPU もインターネットも要らない。非公開にするのは、
    URL が「知っていれば誰でも入れる入口」だから（`url.txt` と同じ扱い）。
    """
    name = slug.rsplit("/", 1)[-1]
    return {
        "id": slug,
        "title": name,
        "code_file": "main.py",
        "language": "python",
        "kernel_type": "script",
        "is_private": True,
        "enable_gpu": False,
        "enable_tpu": False,
        "enable_internet": False,
        "keywords": [],
        "dataset_sources": [],
        "kernel_sources": [],
        "competition_sources": [],
        "model_sources": [],
    }


def kc_beacon_body(url, stage="", updated=None):
    """掲示板の main.py の中身。"""
    return _BEACON_BODY.format(
        url=url, stage=stage, updated=time.time() if updated is None else updated)


def kc_beacon_publish(url, stage="", slug="", timeout=180):
    """掲示板を更新する。戻り値は `(ok, detail)`。

    `detail` は成功なら `"pushed v3 (platform)"` のような内訳、失敗なら理由。
    **例外は投げない**（配信を止めない）。呼び出し側は成否を `result` に残す。
    """
    s = kc_beacon_slug(slug)
    if not s:
        return False, "beacon の slug が無い（KC_BEACON_SLUG か引数で渡す）"
    if "/" not in s:
        return False, f"slug は owner/name の形で渡す: {s!r}"
    try:
        KaggleApi = kc_kaggle_api()
    except Exception as e:
        return False, f"kaggle パッケージが無い（{type(e).__name__}: {e}）"

    d = tempfile.mkdtemp(prefix="beacon-")
    try:
        with open(os.path.join(d, "main.py"), "w", encoding="utf-8") as f:
            f.write(kc_beacon_body(url, stage))
        with open(os.path.join(d, "kernel-metadata.json"), "w",
                  encoding="utf-8") as f:
            json.dump(kc_beacon_meta(s), f, indent=2, ensure_ascii=False)

        api = KaggleApi()
        try:
            api.authenticate()
        except Exception as e:
            return False, (f"Kaggle 認証が無い（{type(e).__name__}: {e}）"
                           " — カーネル外からは ~/.kaggle/kaggle.json が要る")
        try:
            resp = api.kernels_push(d)
        except SystemExit as e:                      # kaggle は終了コードで返す
            return False, f"kernels_push が終了した（SystemExit {e.code}）"
        except Exception as e:
            return False, f"kernels_push 失敗（{type(e).__name__}: {e}）"
        ver = getattr(resp, "version_number", None)
        return True, f"pushed v{ver} ({s})"
    finally:
        shutil.rmtree(d, ignore_errors=True)


def kc_beacon_parse(text):
    """掲示板のソースから `url` / `stage` / `updated` を取り出す。

    `exec` しない。中身は自分で書いたものだが、読む側（VM）にとっては
    ネットワーク越しの入力なので、評価せずに読む。
    """
    out = {"url": "", "stage": "", "updated": None}
    keys = {"URL": "url", "STAGE": "stage", "UPDATED": "updated"}
    for line in (text or "").splitlines():
        s = line.strip()
        for key, field in keys.items():
            if not s.startswith(key + " ="):
                continue
            raw = s.split("=", 1)[1].strip()
            if field == "updated":
                try:
                    out[field] = float(raw)
                except ValueError:
                    out[field] = None
            else:
                out[field] = raw.strip("\"'")
    return out


def kc_beacon_is_fresh(data, since, slack=5.0):
    """`since` より後に発行されたものだけを本物とみなす。

    2026-11-01 に**古いテスト URL を掴む**事故が出ている。push の前後で
    スタンプが変わったことを確かめないと、「まだ前の値」を新しい値として
    読んでしまう。`slack` は VM と Kaggle の時計のずれぶん。
    """
    u = (data or {}).get("updated")
    if not isinstance(u, (int, float)):
        return False
    return u >= (float(since) - float(slack))


def kc_beacon_pull_cli(slug, dest, exe="", timeout=180):
    """`kaggle kernels pull` を**CLI で**実行する。戻り値は `(ok, detail)`。

    Python パッケージの `kaggle` が import できない環境のための逃げ道。
    手元の python3（3.13）には入っておらず、`kaggle` CLI だけが 3.9 の
    user site に入っている、という状態で実際に踏んだ（`cli beacon` が
    `No module named 'kaggle'` で死んでいた）。カーネル内では使われない。
    """
    import subprocess                      # 束ねたときの F811 を避けて局所 import
    cmd = [exe or shutil.which("kaggle") or "", "kernels", "pull", slug, "-p", dest]
    if not cmd[0]:
        return False, "kaggle パッケージも kaggle CLI も見つからない"
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        return False, f"kaggle CLI の起動に失敗（{type(e).__name__}: {e}）"
    if p.returncode != 0:
        tail = (p.stderr or p.stdout or "").strip().splitlines()[-1:] or [""]
        return False, f"kaggle CLI が {p.returncode} で終了（{tail[0][:200]}）"
    return True, "kaggle CLI"


def kc_beacon_pull(slug="", dest=None, timeout=180, exe=""):
    """掲示板を引いて読む。戻り値は `(ok, detail, data)`。

    `data` は `kc_beacon_parse` の結果（失敗なら空）。
    `exe` は CLI 経由に落ちたときに使う `kaggle` のパス（`cli push` と同じ
    解決順を呼び出し側が渡す）。空なら PATH から探す。
    """
    s = kc_beacon_slug(slug)
    if not s:
        return False, "beacon の slug が無い", {}
    d = dest or tempfile.mkdtemp(prefix="beacon-pull-")
    try:
        # まず Python パッケージ。駄目なら CLI に落ちる（理由は捨てない）。
        why_api = ""
        try:
            KaggleApi = kc_kaggle_api()
        except Exception as e:
            why_api = f"{type(e).__name__}: {e}"
        else:
            try:
                api = KaggleApi()
                api.authenticate()
                api.kernels_pull(s, path=d)
                why_api = ""
            except SystemExit as e:          # kaggle は終了コードで返す
                why_api = f"kernels_pull が終了した（SystemExit {e.code}）"
            except Exception as e:
                why_api = f"kernels_pull 失敗（{type(e).__name__}: {e}）"
        if why_api:
            ok, detail = kc_beacon_pull_cli(s, d, exe=exe, timeout=timeout)
            if not ok:
                return False, f"API でも CLI でも引けなかった（API: {why_api} / CLI: {detail}）", {}
        # 引いたファイル名は版によって変わる（main.py / <name>.py）ので、
        # 中身で選ぶ。UPDATED を含む .py が掲示板。
        for root, _dirs, files in os.walk(d):
            for fn in files:
                if not fn.endswith(".py"):
                    continue
                with open(os.path.join(root, fn), encoding="utf-8",
                          errors="replace") as f:
                    text = f.read()
                if "UPDATED" in text:
                    data = kc_beacon_parse(text)
                    return True, f"pulled {fn}", data
        return False, "pull した中に掲示板（UPDATED を含む .py）が無い", {}
    finally:
        if dest is None:
            shutil.rmtree(d, ignore_errors=True)



---
### ⚙️ 設定（GPU検出 → モデル・コンテキスト決定）


In [ ]:
# ============================================================
# Step 2: Ollama 配信カーネルの設定
# ============================================================
# ビルダーが「アンダースコア 2 つで囲んだ大文字名」を置換して埋め込む。
# notebook でも script でも
# この 1 ファイルが唯一の定義（両方の生成物が同じ定数を読む）。
import os

MODEL_NAME = "orcarouter/Qwen3.8-27B-Uncensored:latest"
MODEL_DISPLAY = "Qwen3.8-27B-Uncensored"
ENGINE_DISPLAY = "Ollama"
ENGINE_LABEL = "ollama"

# Dataset のマウント先は共通ランタイム（runtime/storage.py）に一本化。
# 旧 /kaggle/input/<slug> と新 /kaggle/input/datasets/<owner>/<slug> の
# 両対応・空マウント回避は kc_find_dataset が持つ（ビルダー側で書かない）。
DATASET_PATH = kc_find_dataset("my-ollama-orcarouter-qwen38-27b")
PATCH_PATH = kc_find_dataset("my-ollama-orcarouter-qwen38-27b-patch")
# cloudflared 専用 Dataset。フォルダ構造は flatten されることがあるので
# インストール側で bin/ の有無両方を探索する。
CF_PATH = kc_find_dataset("my-cloudflared-bin")
OLLAMA_MODELS = "/kaggle/tmp/ollama_models"
OLLAMA_BIN = "/usr/local/bin/ollama"

# ---- ポート ----
# Ollama 本体は 11434 のまま、公開側はプロキシが 11435 で受ける。
PORT = UPSTREAM_PORT = 11434
PROXY_PORT = 11435
PROXY_CHECK_PATH = "/api/tags"
PROXY_EXTRA_ARGS = ()
# トンネル種別（ビルダーが埋める）。インストール側が
# 「cloudflared を復元するか」を分岐するために使う。
TUNNEL_KIND = "cloudflared"

# Basic 認証プロキシのソース（base64）。ビルダーが auth_proxy.py を埋め込む。
PROXY_SRC_B64 = "..."
# チャット UI（proxy が / で返す）
CHAT_SRC_B64 = "..."
CHAT_PATH = "/usr/local/bin/llama-chat.html"
PROXY_PATH = "/usr/local/bin/auth_proxy.py"

# 認証は既定で無効。API_PASS が入っているときだけ proxy が有効化する。
# cloudflared の quick tunnel に Basic 認証は無いので、UI を開いただけで
# 使えることを優先している（URL を知れば誰でも使える点は許容）。
PROXY_AUTH = {
    "API_USER": os.environ.get("API_USER", "") or "ollama",
    "API_PASS": os.environ.get("API_PASS", ""),
}


# ---- GPU / コンテキスト長 ----
_gpu_names, _vram_each = kc_detect_gpu()
VRAM_TOTAL = sum(_vram_each)          # ← 合計で判定する（per-GPU ではない）
print(f"GPU      : {_gpu_names}")
print(f"VRAM     : {_vram_each} MiB each  →  合計 {VRAM_TOTAL} MiB")

if VRAM_TOTAL == 0:
    raise RuntimeError("GPU が見つかりません。Accelerator を 'GPU T4 x2' に設定してください。")

# 実測: full attention 17層 × kv_head 4 × (key 256 + value 256) × 2B = 68 KiB/token (f16)
#       q8_0 KV ならその半分 = 34 KiB/token
MODEL_GIB = 16.52
KV_KIB_PER_TOK_Q8 = 34.0

# 実測（kaggle/bench/）: 32K=13.7 tps / 131K=13.7 tps / 262K=3.7 tps。
# 131K までは速度が落ちない。262K で 1/3.7 になるので既定は 131K。
CTX_CANDIDATES = [131072, 65536, 32768, 16384, 8192]
# 予算（実効 VRAM の 88%）と候補の当てはめは runtime/proc.py に一本化。
CONTEXT_LENGTH, budget_gib = kc_context_budget(
    VRAM_TOTAL, MODEL_GIB, KV_KIB_PER_TOK_Q8, CTX_CANDIDATES)
print(f"予算     : {budget_gib:.1f} GiB（実効 VRAM の 88%）")

kv_used = KV_KIB_PER_TOK_Q8 * CONTEXT_LENGTH / 1024 / 1024
print(f"コンテキスト: {CONTEXT_LENGTH:,} tokens")
print(f"  モデル {MODEL_GIB} GiB + KV(q8_0) {kv_used:.2f} GiB = {MODEL_GIB + kv_used:.2f} GiB")

# ---- アカウント（ビーコンの slug に使う。config.env の KAGGLE_OWNER）----
KAGGLE_OWNER = "yamahei21python"

# ---- キープアライブ ----
MAX_MINUTES = 480                 # セル開始からの合計上限（Kaggle 9h より短く）
KEEPALIVE_RESET_MINUTES = 60      # チャット送信ごとにリセットする時間（分）
TUNNEL_CHECK_EVERY = 10           # 公開URL疎通は 10 分ごと（外部リクエスト削減）

# ---- 死活監視 / 後片付けの能力表（共有セルが読む）----
ENGINE_LOG = "/tmp/ollama.log"
# ollama は -f だと子の `llama-server -m ...ollama_models...` まで全一致して
# 偽 alive になる。プロセス名完全一致で見る。proxy は argv で見る。
ENGINE_PROCS = [("ollama", "exact"), ("auth_proxy.py", "argv")]
CLEANUP_TARGETS = [
    ("-x cloudflared", "cloudflared"),
    ("-f auth_proxy.py", "auth_proxy"),
    ("-x ollama", "ollama"),
    ("-f llama-server", "llama-server"),
]


def engine_cleanup():
    """VRAM を返してから落とす（OLLAMA_KEEP_ALIVE=-1 で固定されるため）。

    これをやらないと 23 GiB が VRAM に残ったままカーネルが COMPLETE になり、
    GPU を掴み続ける（実測: expires_at が 2319 年になる）。プロセスを落としても
    Kaggle 側の枠開放は保証されないが、少なくとも VRAM は即座に返る。
    """
    unloaded = kc_unload_model(PORT, MODEL_NAME)
    kc_wait_vram_free(PORT)
    return unloaded


# ---- Ollama の起動環境変数 ----
# 起動セルと確認セルが共有するので、ここで 1 回だけ定義する。
# OLLAMA_CONTEXT_LENGTH : コンテキスト長（未設定だと 4096 に落ちる）
# OLLAMA_KV_CACHE_TYPE  : KV を q8_0 に量子化して VRAM を半分に
# OLLAMA_FLASH_ATTENTION: 有効化
# OLLAMA_KEEP_ALIVE     : -1 で常時ロード維持
OLLAMA_ENV = {
    **os.environ,
    "OLLAMA_MODELS": OLLAMA_MODELS,
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_ORIGINS": "*",
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT_LENGTH),
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_FLASH_ATTENTION": "1",
    # 既定 5分（stall 検出）。v9 は load 198s で通過したが v10 は超過して
    # "timed out waiting for llama-server to start" → 500 になった。
    # mmap 読込はページキャッシュ次第で揺れるので 10分に緩和（warmup 900s 未満）。
    "OLLAMA_LOAD_TIMEOUT": "600",
    "OLLAMA_NUM_PARALLEL": "1",   # qwen35 は並列が blocklist されているため 1
}

# ---- 検証成果物の永続化 ----
# `kaggle kernels output` が zip 化するのは **/kaggle/working** のだけ。
# /output は公式の永続先ではなく、作れても zip に載らないことがある
# （= 保存したつもりで証跡ゼロ）。だから両方へ書く。
OUT_DIRS = KC_OUT_DIRS
save_output = kc_save_output
fail = kc_fail

print()
print("=" * 56)
print(f"  認証        : {'OFF（URL を知れば誰でも利用可）' if not PROXY_AUTH['API_PASS'] else 'ON'}")
print(f"  公開ポート  : {PROXY_PORT}（Ollama 本体は {PORT}）")
print("=" * 56)


---
### 📡 Ollama / cloudflared インストール（Dataset キャッシュ優先）


In [ ]:
# バイナリ復元: Dataset キャッシュ優先、不足分のみ DL
import os, shutil, subprocess

BIN_DIR = "/usr/local/bin"
LIB_DIR = "/usr/local/lib/ollama"
BIN_CACHE = os.path.join(PATCH_PATH, "bin")
LIB_CACHE = os.path.join(PATCH_PATH, "lib", "ollama")
MIN_SIZE = 1024 * 1024  # 1MB 未満は壊れているとみなす
# llama-server は例外。Linux 版は 15 KB の薄いラッパーで、
# 実体は libllama-server-impl.so を dlopen する（1MB 判定だと誤って弾く）
MIN_SIZE_LLAMA = 4096

LLAMA_SERVER_DST = os.path.join(LIB_DIR, "llama-server")
# install.sh は llama-server を lib/ollama/ に置く。patch Dataset もその配置で
# 作ってあるので lib 側を第一候補、旧配置の bin/ 側も後方互換で見る。
# サイズ検証を先に行う: 0B/切れファイルが良品を shadow すると missing 判定になり
# install.sh が走って ollama が上書きされる（version 固定が崩れる）。
LLAMA_SERVER_CACHE = None
for _c in (os.path.join(LIB_CACHE, "llama-server"),
           os.path.join(BIN_CACHE, "llama-server")):
    if os.path.isfile(_c) and os.path.getsize(_c) >= MIN_SIZE_LLAMA:
        LLAMA_SERVER_CACHE = _c
        break

restored, missing = [], []


def _find_cached(name):
    """バイナリのキャッシュ位置を探す。見つからなければ None。

    cloudflared は専用 Dataset（my-cloudflared-bin）から復元する。
    Dataset 側のフォルダ構造は flatten されることがあるので、
    `bin/` あり / なし の両方を試してから、既存の patch Dataset に流す。
    """
    candidates = [os.path.join(BIN_CACHE, name)]
    if name == "cloudflared":
        candidates = [
            os.path.join(CF_PATH, "cloudflared"),
            os.path.join(CF_PATH, "bin", "cloudflared"),
        ] + candidates
    for src in candidates:
        if os.path.isfile(src) and os.path.getsize(src) >= MIN_SIZE:
            return src
    return None


# --- ollama / cloudflared ---
# cloudflared は Dataset から復元する。トンネルの種類で分岐しない。
for name in ["ollama", "cloudflared"]:
    dst = os.path.join(BIN_DIR, name)
    src = _find_cached(name)
    if src:
        shutil.copy2(src, dst)
        subprocess.run(["chmod", "+x", dst])
        restored.append(f"{name} <- {src}")
    else:
        missing.append(name)

# --- llama-server（Ollama 0.5+ で必須）---
if LLAMA_SERVER_CACHE and os.path.getsize(LLAMA_SERVER_CACHE) >= MIN_SIZE_LLAMA:
    os.makedirs(LIB_DIR, exist_ok=True)
    shutil.copy2(LLAMA_SERVER_CACHE, LLAMA_SERVER_DST)
    subprocess.run(["chmod", "+x", LLAMA_SERVER_DST])
    restored.append("llama-server")
else:
    missing.append("llama-server")

# --- lib/ollama（CUDA バックエンド）---
# これが無いと llama-server が libggml-cuda.so を dlopen できず、
# エラーを出さずに CPU 推論へ落ちる（実測 0.9 tps / VRAM 0 MiB）。
if os.path.isdir(LIB_CACHE):
    os.makedirs(LIB_DIR, exist_ok=True)
    n_lib = 0
    for root, _dirs, files in os.walk(LIB_CACHE):
        rel = os.path.relpath(root, LIB_CACHE)
        dst_dir = LIB_DIR if rel == "." else os.path.join(LIB_DIR, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for fn in files:
            src_p = os.path.join(root, fn)
            dst_p = os.path.join(dst_dir, fn)
            shutil.copy2(src_p, dst_p)
            # copy2 は mode を保存する。tar 経由の Dataset では x ビットが全部
            # 落ちる（v13 実測 → fork/exec permission denied）。root 実行の
            # os.access(X_OK) は「x ビットが1つでもあれば True」なので、全部
            # 落ちたケースを判定できない。lib/ollama の中身は .so と
            # llama-server だけ＝exec ビット過剰でも無害なので無条件 0755。
            os.chmod(dst_p, 0o755)
            n_lib += 1
    restored.append(f"lib/ollama ({n_lib} files)")
else:
    missing.append("lib/ollama")

# lib walk が llama-server を上書きするので、実行ビットは walk の後に確定させる
if os.path.isfile(LLAMA_SERVER_DST):
    os.chmod(LLAMA_SERVER_DST, 0o755)

# CUDA バックエンドの実在確認（これが GPU 推論の前提）
CUDA_LIB = os.path.join(LIB_DIR, "cuda_v12", "libggml-cuda.so")
print(f"{'✅' if os.path.isfile(CUDA_LIB) else '⚠ '} CUDA バックエンド: {CUDA_LIB}")

print(f"✅ キャッシュ復元: {', '.join(restored) if restored else 'なし'}")
print(f"⬇️  DL が必要  : {', '.join(missing) if missing else 'なし'}")

# --- 不足分だけダウンロード ---
# ollama 本体が在っても llama-server / CUDA が欠けている限り install.sh は必要
# （patch 側に llama-server が無いため v8 で欠落を実測。無いと検証で即 fail）。
# install.sh が ollama を上書きする点は許容 — 上書きが機能不全になる风险より、
# llama-server 欠落で落ちる方を避ける。patch に llama-server 入れるのが本筋。
if missing:
    need_ollama = ("ollama" in missing) or ("llama-server" in missing) or ("lib/ollama" in missing)
    procs = []

    if need_ollama:
        # zstd は ollama の .tar.zst 展開に必要
        if subprocess.run(["dpkg", "-s", "zstd"], capture_output=True).returncode != 0:
            subprocess.run("apt-get install -y -qq zstd", shell=True, check=False)
        print("⬇️  ollama をインストール中...")
        procs.append(subprocess.Popen(
            "curl -fsSL --connect-timeout 15 https://ollama.com/install.sh | sh",
            shell=True,
        ))

    if "cloudflared" in missing:
        print("⬇️  cloudflared をダウンロード中...")
        procs.append(subprocess.Popen(
            "curl -fsSL --connect-timeout 15 -o /usr/local/bin/cloudflared "
            f"{kc_cloudflared_url()} && "
            "chmod +x /usr/local/bin/cloudflared",
            shell=True,
        ))

    for p in procs:
        # 無限待ちはハングで notebook が止まり成果物が残らない。
        # install.sh（tar.zst 展開込み）・curl とも最大10分まで。
        try:
            p.wait(timeout=600)
        except subprocess.TimeoutExpired:
            p.kill()
            p.wait()
            fail("バイナリ取得がタイムアウトしました（10分）")

# --- 検証 ---
print()
ok = True
_targets = [(f"{BIN_DIR}/ollama", MIN_SIZE),
            (f"{BIN_DIR}/cloudflared", MIN_SIZE),
            (LLAMA_SERVER_DST, MIN_SIZE_LLAMA),
            (CUDA_LIB, MIN_SIZE)]
for path, min_sz in _targets:
    exists = os.path.isfile(path)
    size = os.path.getsize(path) if exists else 0
    mark = "✅" if exists and size >= min_sz else "❌"
    if mark == "❌":
        ok = False
    print(f"{mark} {path}  ({size:,} B)")

# 実行ビットは tar 経由の Dataset で落ちる（v13 実測: fork/exec permission denied）。
# サイズだけ見ていると通ってしまい、起動時に 500 になるので明示的に確認する。
# root 実行では os.access(X_OK) が「x ビットが1つでもあれば True」になるため、
# mode ビットで 3 桁すべて（0o111）が立っているかを見る。
_exes = [f"{BIN_DIR}/ollama", f"{BIN_DIR}/cloudflared", LLAMA_SERVER_DST]
for exe in _exes:
    if not os.path.isfile(exe):
        continue
    if os.stat(exe).st_mode & 0o111 != 0o111:
        try:
            os.chmod(exe, 0o755)
            print(f"🔧 実行ビット付与: {exe}")
        except OSError as e:
            print(f"❌ chmod 失敗: {exe} ({e})")
            ok = False
    if os.stat(exe).st_mode & 0o111 != 0o111:  # 事後検証（noexec 等を検出）
        print(f"❌ 実行ビット未設定: {exe}")
        ok = False

if not ok:
    fail("バイナリの準備に失敗しました（cloudflared は Dataset から、"
         "それ以外は GitHub / ollama.com から取る想定）")

print()
# --version が通る = 実行ビットの end-to-end 証明。returncode まで見る
# （exec 失敗だと stdout が空のまま素通りする）。
_ver_exes = [f"{BIN_DIR}/ollama", f"{BIN_DIR}/cloudflared"]
for _exe in _ver_exes:
    _r = subprocess.run([_exe, "--version"], capture_output=True, text=True)
    print((_r.stdout or _r.stderr).strip())
    if _r.returncode != 0:
        fail(f"{_exe} 起動失敗 (returncode={_r.returncode})")

# --- Basic 認証プロキシを書き出す（base64 埋め込み）---
# PROXY_PATH は設定セルの定義を使う（二重管理しない）
import base64 as _b64
with open(PROXY_PATH, "wb") as f:
    f.write(_b64.b64decode(PROXY_SRC_B64))
print(f"✅ auth_proxy.py を書き出し: {PROXY_PATH} "
      f"({os.path.getsize(PROXY_PATH):,} B)")

with open(CHAT_PATH, "wb") as f:
    f.write(_b64.b64decode(CHAT_SRC_B64))
print(f"✅ llama-chat.html を書き出し: {CHAT_PATH} "
      f"({os.path.getsize(CHAT_PATH):,} B)")


---
### 🌐 リバースプロキシ起動（UI 配信 + 同一オリジン化）

推論サーバは非公開、公開側は認証プロキシが受けます。


In [ ]:
# リバースプロキシを起動（UI を / で配信し、API を同一オリジンにする）
# 起動・待機・認証なし検査の実装は共通ランタイム（runtime/proxy.py）。
# 中継先・追加引数・認証は settings セルが定義する（PROXY_* / UPSTREAM_PORT）。
import os

# 認証は既定で無効。PROXY_AUTH に API_PASS が入っているときだけ有効になる。
proxy_env = {**os.environ, **PROXY_AUTH}
proxy_proc = kc_start_proxy(
    PROXY_PORT, f"http://127.0.0.1:{UPSTREAM_PORT}", CHAT_PATH, PROXY_PATH,
    extra_args=PROXY_EXTRA_ARGS, env=proxy_env, fail=fail,
)

# 推論サーバ起動前なので 5xx は正常。401/403 が返るなら認証の設定ミス。
# 完全判定（200 / streaming / OpenAI 互換）は公開URL経由の verify セルへ一本化。
kc_assert_no_auth(PROXY_PORT, path=PROXY_CHECK_PATH, fail=fail)


---
### 🔥 Ollama サーバー起動


In [ ]:
# Dataset を展開 → Ollama 起動（コンテキスト長・KV 量子化を明示）
import os, shutil, subprocess, time

os.makedirs(OLLAMA_MODELS, exist_ok=True)

# --- モデルデータの準備 ---
# blobs は「ディレクトリごと symlink」ではなく「実ディレクトリ + ファイル単位
# symlink」にする。dir ごと symlink だと OLLAMA_MODELS 配下が read-only
# （/kaggle/input は読み取り専用マウント）になり、欠けた blob の補完 pull が
# `read-only file system` で即死する（v8 実測）。
import glob as _g, json as _json

def _stage(name, extra_roots=()):
    """DATASET_PATH/name 以下のファイルを実dirとして配置（file symlink）。

    patch Dataset の blobs（config blob sha256-cbe6ad4… 214B が入っている）
    も合流させる。これが無いと `ollama show` が失敗して毎回 pull 誘発。
    """
    dst = os.path.join(OLLAMA_MODELS, name)
    os.makedirs(dst, exist_ok=True)
    n = 0
    for root in (DATASET_PATH,) + tuple(extra_roots):
        base = os.path.join(root, name)
        if not os.path.isdir(base):
            continue
        for p in sorted(_g.glob(os.path.join(base, "**"), recursive=True)):
            if not os.path.isfile(p):
                continue
            rel = os.path.relpath(p, base)
            d = os.path.join(dst, rel)
            # manifests は「実ファイル」のみ正。symlink は v11 の残骸なので
            # 作り直す（skip すると同一セッションの再実行で修正が効かない）
            if name == "manifests":
                if os.path.lexists(d) and not os.path.islink(d) and os.path.isfile(d):
                    continue
                if os.path.lexists(d):
                    os.remove(d)
                os.makedirs(os.path.dirname(d), exist_ok=True)
                # ollama は manifest 自体を symlink 検査し、対象が sha256 blob で
                # ないと拒否する（v11 実測 "manifest symlink target ... is not a
                # sha256 blob" → show 失敗 → pull 誘発）→ 1KB 級なので実体コピー
                shutil.copyfile(p, d)
                n += 1
                continue
            # blobs: 有効な symlink/実ファイルだけスキップ、壊れていれば作り直す
            # （lexists は壊れた symlink も True なので isfile で見分ける）
            if os.path.lexists(d) and os.path.isfile(d):
                continue
            if os.path.lexists(d):
                os.remove(d)
            os.makedirs(os.path.dirname(d), exist_ok=True)
            try:
                os.symlink(p, d)
            except OSError:
                # 16 GiB を /kaggle/tmp へ複製すると disk full。巨大ファイルは
                # 複製せず落とす（symlink 不可 = 環境異常なので即中断が正解）
                if os.path.getsize(p) > (1 << 30):
                    raise
                shutil.copy2(p, d)
            n += 1
    return dst, n

if os.path.isdir(DATASET_PATH) and os.listdir(DATASET_PATH):
    print("📁 Dataset 検出 → モデルデータを準備中...")
    # 失敗時は例外のまま死なせず、理由を FAILED.txt に残す
    try:
        _, n_blob = _stage("blobs", extra_roots=(PATCH_PATH,))
        _, n_man = _stage("manifests")
    except OSError as _e:
        fail(f"モデルデータ配置に失敗: {type(_e).__name__}: {_e}")
    print(f"✅ blobs {n_blob} 件 / manifests {n_man} 件を配置（patch の config blob 含む）")

    # manifest が要求する blob の充足を先に点検。欠けたまま `ollama show` を
    # 叩くと pull が走り、v8 の失敗経路（16.5 GiB 再DL）へ戻る。
    _repo, _tag = MODEL_NAME.rsplit(":", 1)
    _man = os.path.join(OLLAMA_MODELS, "manifests", "registry.ollama.ai",
                        *_repo.split("/"), _tag)
    if os.path.isfile(_man):
        with open(_man) as _f:
            _md = _json.load(_f)
        _need = [_md.get("config", {})] + _md.get("layers", [])
        _miss = [e["digest"] for e in _need
                 if e.get("digest") and not os.path.lexists(os.path.join(
                     OLLAMA_MODELS, "blobs",
                     "sha256-" + e["digest"].split(":", 1)[1]))]
        if _miss:
            print(f"⚠ 欠落 blob {len(_miss)} 件 → pull フォールバックの可能性: {_miss[:5]}")
        else:
            print("✅ manifest 要求 blob 全数充足 → pull 不要")
    else:
        print(f"⚠ manifest が見つからない: {_man}")
else:
    print(f"⚠ Dataset が見つかりません: {DATASET_PATH}")
    print("  → Add Data で Dataset を追加するか、下で直接 pull します。")

# --- 環境変数 / 実体パス ---
# `env` と `OLLAMA_BIN` は設定セル（00_settings.py）が持つ。
# 確認セル・ウォームアップセルも同じ値を見るので、ここでは再定義しない。

# --- 既存プロセスを掃除 ---
# 前回セッションの孤児 llama-server が残っているとポート/VRAMを
# 奪って本番と競合する。SIGTERM → 待機 → 残存分は SIGKILL。
subprocess.run(["pkill", "-TERM", "-f", "llama-server"], capture_output=True)
time.sleep(3)
subprocess.run(["pkill", "-9", "-f", "llama-server"], capture_output=True)
subprocess.run(["pkill", "-9", "ollama"], capture_output=True)
time.sleep(2)

with open(ENGINE_LOG, "w") as log:
    proc = subprocess.Popen([OLLAMA_BIN, "serve"], env=OLLAMA_ENV,
                            stdout=log, stderr=log)
print(f"🔄 Ollama PID: {proc.pid}")

# --- 起動待機 ---
for i in range(60):
    r = subprocess.run([OLLAMA_BIN, "list"], env=OLLAMA_ENV, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"✅ Ollama サーバー起動完了（{i+1}s）")
        break
    time.sleep(1)
else:
    print(open(ENGINE_LOG).read()[-3000:])
    fail("Ollama サーバー起動失敗", "/tmp/ollama.log")

# --- モデル確認 ---
r = subprocess.run([OLLAMA_BIN, "show", MODEL_NAME], env=OLLAMA_ENV,
                   capture_output=True, text=True)
if r.returncode != 0:
    # なぜ show が失敗したかを残さないと「全数充足なのに pull」が再現不能になる
    print(f"⚠ show 失敗（stderr）: {r.stderr.strip()[:500]}")
    print("⏳ Dataset にモデルが無いため pull します（16.5 GiB / 10〜30分）")
    # pull 前に空き容量を確認。16.5 GiB + 退避バッファが無いと
    # 途中で ENOSPC になり、長い待機の後に初めて失敗するのを防ぐ。
    _du = shutil.disk_usage("/kaggle/tmp")
    _need = 18 * (1 << 30)   # 16.5 GiB + 余裕
    print(f"   disk /kaggle/tmp: free={_du.free/(1<<30):.1f} GiB "
          f"(need ~{_need/(1<<30):.0f} GiB)")
    if _du.free < _need:
        fail(f"/kaggle/tmp 空き不足: {_du.free/(1<<30):.1f} GiB "
             f"< 必要 {_need/(1<<30):.0f} GiB")
    # 失敗しても例外のまま死なず、証跡を残してから止める
    # （進捗はセル出力へそのまま流す。pull のエラー表示もそこに残る）
    try:
        pull = subprocess.run([OLLAMA_BIN, "pull", MODEL_NAME], env=OLLAMA_ENV,
                              timeout=7200)
    except subprocess.TimeoutExpired:
        fail("モデル pull が 7200s でタイムアウトしました")
    if pull.returncode != 0:
        fail("モデル pull に失敗しました（詳細はこのセルの出力）")
    print("✅ モデルダウンロード完了")
else:
    print(f"✅ モデル {MODEL_NAME} は利用可能")

print()
print(subprocess.run([OLLAMA_BIN, "list"], env=OLLAMA_ENV,
                     capture_output=True, text=True).stdout)


---
### 🔥 ウォームアップ（初回ロードは数分かかります）


In [ ]:
# モデルを VRAM にロードして疎通確認
# 初回は 2〜5 分かかります（16.5 GiB をロード）
import json, time, urllib.request

def _post(path, payload, timeout=900):
    req = urllib.request.Request(
        f"http://localhost:{PORT}{path}",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

print(f"⏳ ウォームアップ中（コンテキスト {CONTEXT_LENGTH:,}）...")
t0 = time.time()
try:
    resp = _post("/api/chat", {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say OK."}],
        "stream": False,
        # 16 だと thinking（推論トークン）だけで予算を使い切り、
        # content が空のまま返る（実測 reply=''）。生成証明には十分な余量を。
        "options": {"num_predict": 128, "temperature": 0},
    })
    dt = time.time() - t0
    print(f"✅ 応答あり（{dt:.1f}s）")
    print(f"   load : {(resp.get('load_duration') or 0)/1e9:.1f}s")
    print(f"   reply: {resp.get('message', {}).get('content', '')!r}")
    ec = resp.get("eval_count") or 0
    ed = (resp.get("eval_duration") or 1) / 1e9
    print(f"   速度 : {ec/ed:.1f} tokens/s" if ec else "   速度 : N/A")
    if not (resp.get("message", {}).get("content")
            or resp.get("message", {}).get("thinking")):
        # HTTP 成功だが生成物が空 = verify [2] と同じ症状の予兆なので可視化する
        print("   ⚠ content/thinking が空（verify [2] で判定されます）")
    # GPU 常駐検査。CPU フォールバックでも HTTP 200 は返るので、
    # 通信だけ見ていると 3/3 PASS のまま静かに劣化する。
    import subprocess as _sp
    _ps = _sp.run([OLLAMA_BIN, "ps"], env=OLLAMA_ENV,
                  capture_output=True, text=True).stdout.strip().splitlines()
    print("   ps :", " | ".join(l.strip() for l in _ps[1:] if l.strip()) or "(空)")
    _psrow = " ".join(l for l in _ps[1:] if "%" in l)
    if _psrow and "GPU" not in _psrow:
        print("   ⚠ プロセッサ表示に GPU が無い → CPU 実行の可能性")
    # ollama ps の CONTEXT 列はカンマ無し（131072）。正規化して比較する。
    if _psrow and str(CONTEXT_LENGTH) not in _psrow.replace(",", ""):
        print(f"   ⚠ コンテキストが {CONTEXT_LENGTH:,} と異なる（ps 表示: {_psrow}）")
except Exception as e:
    print(f"⚠ ウォームアップ失敗: {e}（LOAD_TIMEOUT={OLLAMA_ENV.get('OLLAMA_LOAD_TIMEOUT')}s）")
    print(open(ENGINE_LOG).read()[-3000:])
    # 握り潰すと verify が「原因不明の FAIL」になるだけなので、ここで止める
    fail(f"ウォームアップ失敗: {type(e).__name__}: {e}", "/tmp/ollama.log")


---
### 🔗 cloudflared トンネル → 公開URL


In [ ]:
# cloudflared quick tunnel 起動 → 公開URL取得
# アカウント不要。authtoken も Secrets も要らない。
import os, subprocess, time

CF_BIN = "/usr/local/bin/cloudflared"
TUNNEL_NAME = "cloudflared"
TUNNEL_LOG = "/tmp/cloudflared.log"
RESTART_CMD = [CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}",
               "--no-autoupdate"]


def probe_url():
    """公開URLを取り直す（keepalive の再起動用）。

    トンネル実装が満たすインターフェース（TUNNEL_NAME / TUNNEL_LOG /
    RESTART_CMD / probe_url）を揃える。揃っていないと keepalive セルが
    NameError になり、トンネル再起動の経路だけ無言で死ぬ（実測）。
    """
    return kc_probe_url("cloudflared", TUNNEL_LOG)


# 前回のトンネルを確実に落とす（多重起動すると URL が混ざる）
subprocess.run(["pkill", "-x", "cloudflared"], capture_output=True)
time.sleep(1)

with open(TUNNEL_LOG, "w") as log:
    cf_proc = subprocess.Popen(RESTART_CMD, stdout=log, stderr=log)
print(f"🔄 cloudflared PID: {cf_proc.pid}")

# ---- 公開URL取得（ログに URL が出るまで待つ）----
public_url = kc_wait_url("cloudflared", TUNNEL_LOG)

if not public_url:
    # ここで止めないと後段が /tmp/public_url.txt なしでエラーになり、
    # どこで失敗したか分からなくなる。原因ログも永続領域に残す。
    print("⚠ 公開URLを取得できませんでした")
    print(open(TUNNEL_LOG).read()[-3000:])
    fail("cloudflared の公開URL取得に失敗しました", TUNNEL_LOG)

# ---- トンネル確立まで待つ ----
# URL が出た直後はまだエッジに伝播しておらず、外から叩くと 000 になる。
# ここで待たないと次のセルの疎通確認が誤って失敗する。
print(f"🌍 URL: {public_url}")
print("⏳ トンネル確立待ち（最大 60s）...")
elapsed = kc_wait_http(f"{public_url}/healthz", attempts=20, interval=3,
                       opener=kc_urlopen_v4)
if elapsed is not None:
    print(f"✅ トンネル確立（{elapsed}s）")
else:
    print("⚠ 確立を確認できませんでした。verify セルで再確認します。")

with open("/tmp/public_url.txt", "w") as f:
    f.write(public_url)
# 完了後に `kaggle kernels output` で取り出す用
save_output("public_url.txt", public_url + "\n")
print()
print("=" * 60)
print(f"  🌍 PUBLIC URL : {public_url}")
print("=" * 60)


---
### 📡 公開URLの配布（ビーコンカーネル）

quick tunnel の URL は毎回変わるので、**取得先だけ固定**します。
同じアカウントの掲示板カーネル（`serve-url-beacon`）の `main.py` を
書き換え、クライアントは `kaggle kernels pull` で URL を読みます。
Secrets も外部サービスも要りません（`notes/url-publishing.md`）。


In [ ]:
# 現在の公開URLをビーコンカーネルに書き出す（取得先の固定）
# Secrets: 不要。カーネルに注入された Kaggle 認証で `kernels_push` する。
# keepalive でトンネルが再起動したときも同じ関数を呼ぶ。
#
# Gist 版（`url.txt` を PATCH する経路）は 2026-11-02 に廃止した。
# Kaggle の Secret は**カーネルごとに紐付ける**必要があり、未紐付けの
# `get_secret()` は `ConnectionError`（＝通信障害に見える）を返すため、
# カーネルを増やすたびに UI 作業が要っていた。ビーコンはその作業ごと不要にする。

# 公開 URL を置く掲示板カーネルの名前（owner は KAGGLE_OWNER）。
# カーネルから push できるのは自分のカーネルだけなので、配信カーネルと
# 同じアカウントに置く。settings で `BEACON_NAME` を定義すれば上書きできる。
BEACON_NAME = globals().get("BEACON_NAME") or "serve-url-beacon"


def publish_url(u):
    """現在の公開URL をビーコンに書く（URL が毎回変わるため）。

    ビーコン（`kc_beacon_publish`）は**Secrets を使わない**経路で、カーネルの
    コードを掲示板にする。失敗しても配信は止めないが、理由は必ず出す
    （2026-11-01 に「URL が 10/28 から一度も更新されていないことに誰も
    気づかない」という失敗をしている）。`notes/url-publishing.md`。
    """
    return _publish_beacon(u)


def _publish_beacon(u):
    """ビーコンを更新する。失敗しても配信は止めない（理由は必ず出す）。"""
    slug = f"{KAGGLE_OWNER}/{BEACON_NAME}" if KAGGLE_OWNER else ""
    try:
        ok, why = kc_beacon_publish(u, stage="serving", slug=slug)
    except NameError:
        # beacon を束ねていないビルド（移行前の notebook）。
        print("⚠ ビーコン未同梱 → 公開 URL を外から読めない")
        return False
    print(("📡 ビーコン更新: " + why) if ok else ("⚠ ビーコン更新失敗（続行）: " + why))
    return ok


publish_url(public_url)
print("📡 ビーコン取得先: kaggle kernels pull " + KAGGLE_OWNER + "/" + BEACON_NAME)


---
### ✅ 公開URL経由の疎通確認（UI / chat / streaming / OpenAI互換 / GPU常駐）


In [ ]:
# Ollama 固有の確認項目（足場は kaggle/serve-cells/verify.py が持つ）
# 呼び出し側: engine_checks(judge, retry, base)
import json, time as _time, urllib.request


def _call(url, timeout=900, stream=False, auth=None):
    headers = {"Content-Type": "application/json"}
    if auth:
        headers["Authorization"] = auth
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Say OK."}],
        "stream": stream,
        # 非ストリームの 16 は thinking だけで枯渇し content が空になる
        # （実測 reply=''）。十分な余量を与える。
        "options": {"num_predict": 128 if not stream else 64, "temperature": 0},
    }
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers=headers, method="POST")
    return urllib.request.urlopen(req, timeout=timeout)


def engine_checks(judge, retry, base, auth=None, direct=False):
    # [1] 認証なしでチャット UI が取れる（/ は 200 で HTML）
    #     UI は proxy が返すもの。smoke（direct=True）は proxy を立てないので
    #     `/` はエンジン自身の応答になる（Ollama は "Ollama is running" の 17 B）。
    #     ここで FAIL にすると「エンジンは正常なのに失敗」と誤判定するため SKIP する。
    print("[1] 認証なしで UI")
    if direct:
        judge(1, "ui", None, "skip: エンジン直（proxy 無し。smoke）")
    else:
        try:
            with retry(lambda: urllib.request.urlopen(f"{base}/", timeout=60)) as r:
                html = r.read()
                # UI 判定は HTML であること + 十分なサイズで行う。
                # 旧実装は b"Kaggle Chat" を見ていたが、HTML 側のタイトル変更で
                # 常に FAIL になる（実測）。文言に依存させない。
                ok1 = (r.status == 200 and len(html) > 10000
                       and html.lstrip()[:15].lower().startswith(b"<!doctype html"))
                judge(1, "ui", ok1, f"HTTP {r.status} {len(html)}B")
        except Exception as e:
            judge(1, "ui", False, f"{type(e).__name__}: {e}")

    # [2] 認証なしで /api/chat が 200 + 生成
    print("[2] /api/chat")
    try:
        with retry(lambda: _call(f"{base}/api/chat", auth=auth)) as r:
            resp = json.loads(r.read())
            reply = (resp.get("message") or {}).get("content", "")
            think = (resp.get("message") or {}).get("thinking", "")
            # 生成証明 = content か thinking か評価トークンのどれかが実在する
            #（thinking モデルは content が空でも生成は成功している）。
            made = bool(reply or think or (resp.get("eval_count") or 0))
            judge(2, "chat-ok", r.status == 200 and made,
                  f"HTTP {r.status} reply={reply!r} "
                  f"eval={resp.get('eval_count')}")
    except Exception as e:
        judge(2, "auth-ok", False, f"{type(e).__name__}: {e}")

    # [3] ストリーミング（chunked がトンネルを通り抜けるか）
    #     ここが通らなければ token 逐次出力 = 通常の利用形態が壊れる。
    print("[3] ストリーミング")
    try:
        times = []
        # 接続確立を先に終えてから計測を始める（再試行の sleep が first/total に
        # 混ざると逐次判定が偽 FAIL になる）
        resp3 = retry(lambda: _call(f"{base}/api/chat", timeout=600, stream=True, auth=auth))
        t0 = _time.time()
        with resp3 as r:
            for raw in r:            # 改行が届くまで待つ = 逐次受信になっている証拠
                if not raw.strip():
                    continue
                times.append(_time.time() - t0)
        total = _time.time() - t0
        if len(times) >= 3:
            first = times[0]
            spread = times[-1] - first
            # 到着分布で判定（生成長に依存しない）:
            #  - 全バッファ転送なら first ≈ total（first < total*0.8 が偽）。
            #  - 3 到着以上 & 最初と最後の間隔が 0.2s 超 = 逐次の到着証拠。
            ok = spread > 0.2 and first < total * 0.8
            detail = (f"chunks={len(times)} first={first:.2f}s "
                      f"total={total:.2f}s spread={spread:.2f}s")
        else:
            ok, detail = False, f"chunks={len(times)} first=N/A"
        judge(3, "streaming", ok, detail)
    except Exception as e:
        judge(3, "streaming", False, f"{type(e).__name__}: {e}")

    # [4] OpenAI 互換経路（配布する opencode.json が使うのはこちら）
    print("[4] /v1/chat/completions")
    try:
        req4 = urllib.request.Request(
            f"{base}/v1/chat/completions",
            data=json.dumps({
                "model": MODEL_NAME,
                "messages": [{"role": "user", "content": "Say OK."}],
                "max_tokens": 128,
            }).encode(),
            headers={"Content-Type": "application/json",
                     **({"Authorization": auth} if auth else {})},
            method="POST",
        )
        with retry(lambda: urllib.request.urlopen(req4, timeout=300)) as r:
            st4 = r.status
            cj = json.loads(r.read())
        ctext = "".join((c.get("message") or {}).get("content") or ""
                        for c in cj.get("choices", []))
        usage = cj.get("usage") or {}
        ok4 = st4 == 200 and bool(ctext or usage.get("completion_tokens"))
        judge(4, "openai-compat", ok4,
              f"HTTP {st4} reply={ctext[:40]!r} usage={usage}")
    except Exception as e:
        judge(4, "openai-compat", False, f"{type(e).__name__}: {e}")

    # [5] GPU 常駐 + コンテキスト長（CPU フォールバックの検出）
    #     通信だけ見ていると 3/3 PASS のまま VRAM に載っていないことがある。
    print("[5] GPU 常駐")
    try:
        import subprocess as _sp
        ps_out = _sp.run([OLLAMA_BIN, "ps"],
                         env=OLLAMA_ENV, capture_output=True, text=True,
                         timeout=30).stdout
        # データ行だけに絞る（ヘッダは "GPU" 文字を含まない）。
        # ollama ps の CONTEXT 列はカンマ無し（例: 131072）なので両方正規化して比較。
        rows = [ln.strip() for ln in ps_out.splitlines()[1:] if "%" in ln]
        row = " | ".join(rows)
        # 完全 CPU フォールバックは "100% CPU" で GPU 文字が無い。
        # 分割オフロード（x% GPU/y% CPU）は VRAM 常駐の一部として許容する。
        has_gpu = "GPU" in row
        has_ctx = str(CONTEXT_LENGTH) in row.replace(",", "")
        judge(5, "gpu-resident", bool(row) and has_gpu and has_ctx,
              f"ps={row[:160]!r} ctx_ok={has_ctx}")
    except Exception as e:
        judge(5, "gpu-resident", False, f"{type(e).__name__}: {e}")


In [ ]:
# 公開URL経由の疎通確認（共通の足場）
# 確認項目そのものは engine_checks() が持つ（1 つ前のセルで定義する）。
# 結果は /kaggle/working/verify_result.txt に残す（save_output が /output にも複製）。
# 失敗は raise（検証不能のまま放置しない）。
import json, time as _time, urllib.request, urllib.error

try:
    with open("/tmp/public_url.txt") as f:
        public_url = f.read().strip()
except FileNotFoundError:
    fail("公開URLがありません。トンネルセルを実行してください。")

# ---- トンネル確立の取り直し ----
# tunnel セルでエッジ伝播待ちが終わっていなくても、ここで吸収する。
# 届かなければ [1] がそのまま FAIL になる（握り潰さない）。
# Kaggle VM は AAAA を返すのに IPv6 経路が無いことがあるので IPv4 強制を使う。
_elapsed = kc_wait_http(f"{public_url}/healthz", attempts=10, interval=3,
                        opener=kc_urlopen_v4)
if _elapsed is not None:
    if _elapsed > 3:
        print(f"✅ トンネル確立をここで確認（{_elapsed}s）")
else:
    print("⚠ /healthz に届きません。トンネル未確立の可能性（[1] で判定）")

lines = [
    f"PUBLIC_URL={public_url}",
    f"MODEL={MODEL_NAME}",
    f"TIME={_time.strftime('%Y-%m-%d %H:%M:%S')}",
    "",
]
failures = []


def _judge(idx, label, ok, detail):
    # ok=None は「この条件では判定できない」= SKIP。失敗には数えないが、
    # PASS とも書かない（後から見て「通った」と誤読しないようにする）。
    if ok is None:
        print(f"⏭ {label}: {detail}")
        lines.append(f"[{idx}] {label} -> SKIP: {detail}")
        return
    mark = "✅" if ok else "❌"
    print(f"{mark} {label}: {detail}")
    lines.append(f"[{idx}] {label} -> {'PASS' if ok else 'FAIL'}: {detail}")
    if not ok:
        failures.append(f"[{idx}] {label}: {detail}")


def _with_retry(fn, tries=6, delay=5):
    # 実行時ログで確認した瞬間的障害: URLError [Errno 101] Network is
    # unreachable / [-2] Name or service not known。3回/4秒では吸収できず、
    # 待ちを 5,10,15,20,25 秒と伸ばして合計 75 秒の窓を持つ。
    # HTTP 応答（401/500 等）は「答え」なので再試行しない。
    return kc_retry(fn, tries=tries, delay=delay)


# ---- エンジン固有の確認項目 ----
engine_checks(_judge, _with_retry, public_url)

# ---- レポート ----
report = "\n".join(lines)
print()
print("=" * 60)
print(report)
print("=" * 60)
if not save_output("verify_result.txt", report + "\n"):
    # 保存失敗 = 証跡なし。ログは取れないので成功扱いにしてはいけない。
    failures.append("verify_result.txt を永続領域に保存できなかった")

if failures:
    fail("疎通確認に失敗: " + "; ".join(failures), ENGINE_LOG)
print("✅ すべての確認項目をパス")


---
### 🔌 接続情報

クライアント設定にコピペする値と `opencode.json` を出力します。


In [ ]:
# 接続情報（クライアント側の設定にコピペしてください）
# セル出力・ログは後から取れないので /kaggle/working に保存する（/output にも複製）。
import json

try:
    with open("/tmp/public_url.txt") as f:
        url = f.read().strip()
except FileNotFoundError:
    fail("公開URLがありません。トンネルセルを実行してください。")

# コンテキスト長はエンジンが決める（未定義ならエンジン既定と表示）
_ctx = f"{CONTEXT_LENGTH:,}" if "CONTEXT_LENGTH" in globals() else "エンジン既定"

opencode_cfg = {
    "$schema": "https://opencode.ai/config.json",
    "provider": {
        "kaggle": {
            "npm": "@ai-sdk/openai-compatible",
            "name": f"Kaggle {ENGINE_DISPLAY}",
            "options": {"baseURL": f"{url}/v1"},
            "models": {MODEL_NAME: {"name": MODEL_DISPLAY}},
        }
    },
    "model": f"kaggle/{MODEL_NAME}",
}

buf = [
    "=" * 64,
    f"  🌍 Public URL : {url}",
    f"  💬 チャット UI : {url}/",
    f"  🤖 Model      : {MODEL_NAME}",
    f"  📏 Context    : {_ctx}",
    "  🔓 認証        : なし（URL を知れば誰でも使える）",
    "=" * 64,
    "",
    "📋 curl:",
    f"  curl {url}/v1/models",
    "",
    "📋 Python（OpenAI 互換クライアント）:",
    "  import httpx",
    "  from openai import OpenAI",
    "  client = OpenAI(",
    f"      base_url='{url}/v1',",
    "      api_key='dummy',",
    "  )",
    f"  client.chat.completions.create(model='{MODEL_NAME}', messages=[...])",
    "",
    "📋 opencode.json:",
    json.dumps(opencode_cfg, indent=2, ensure_ascii=False),
    "",
    "⚠ quick tunnel: 稼働保証なし。切れたらトンネルセルを再実行",
    "⚠ 公開URLは実行のたびに変わる。再実行後は接続情報を取り直す",
    "⚠ Kaggle: 9時間でカーネルが停止します",
]

report = "\n".join(buf)
print(report)
if not save_output("connection_info.txt", report + "\n"):
    # 接続情報は唯一の利用手掛かりなので、取れなければ成功にしない
    fail("connection_info.txt を永続領域に保存できませんでした")


---
### 🔋 キープアライブ

有限ループで自動終了します（`KEEPALIVE_RESET_MINUTES` / `MAX_MINUTES`）。

- チャット送信で残り時間がリセットされます（`/tmp/keepalive.conf`）
- 「■ 停止」ボタンで `RUN=0` が書かれ、ループが終了します


In [ ]:
# キープアライブ: 有限ループ（完了させると /kaggle/working の zip を取得できる）
# 無限ループだと実行が完了せず、ログも output も構造的に取れないため有限にしてある。
#
# 維持時間は /tmp/keepalive.conf（proxy がチャット送信ごとに書き直す）で決まる。
# MAX_MINUTES はセル開始からの合計上限。延長を重ねてもこれを超えない
# （超えて Kaggle の実行時間上限に達すると成果物ごと消える）。
#
# エンジン差は settings の能力表だけ:
#   ENGINE_LABEL / ENGINE_PROCS（名前と照合方法）/ CLEANUP_TARGETS / ENGINE_LOG
#   engine_cleanup()（プロセスを落とす前のエンジン固有の後片付け）
import subprocess, time

KEEPALIVE_CONF = "/tmp/keepalive.conf"   # ON/OFF + MINUTES の共有ファイル


def _read_keepalive_conf():
    """RUN=1/0 と MINUTES を読み取り（上限・既定値は共通ランタイムが処理）。"""
    run, minutes, _mtime = kc_read_conf(
        KEEPALIVE_CONF, KEEPALIVE_RESET_MINUTES, MAX_MINUTES, fail=fail,
        quiet_clamp=True)
    return run, minutes


try:
    with open("/tmp/public_url.txt") as f:
        url = f.read().strip()
except FileNotFoundError:
    url = ""


def _states():
    """エンジンのプロセス生死。名前と照合方法は ENGINE_PROCS が持つ。"""
    return {name: kc_proc_state(name, match) for name, match in ENGINE_PROCS}


def _gpu():
    return kc_gpu_usage()


print(f"Public URL: {url}")
RUN, KEEPALIVE_MINUTES = _read_keepalive_conf()
if RUN:
    print(f"🔋 ON。{KEEPALIVE_RESET_MINUTES}分間維持（チャット送信でリセット、■ で停止）")
else:
    print("🛑 OFF（ RUN=0 ）。即時終了します")
    KEEPALIVE_MINUTES = 0
print("完了後: kaggle kernels output <slug> -p <dir> で成果物を取得")
print()


def _conf_mtime():
    """keepalive.conf の更新時刻。proxy がチャット送信ごとに書き直す。"""
    return kc_conf_mtime(KEEPALIVE_CONF)


log = []
st = {name: "unknown" for name, _ in ENGINE_PROCS}
tunnel_st = "unknown"
tunnel_restarted = 0
tunnel_dead = 0
_restarted_now = False
# 1周も回らなかった場合（RUN=0 即時終了）は死活判定をしない。
# ループ未実行だと st が "unknown" のままになり、正常に動いているのに
# fail() する（実測バグ）。
ran_any = False
i = 0
# for ではなく while にする理由: proxy はチャット送信ごとに
# /tmp/keepalive.conf を書き直す。for の range は開始時に固定されるため、
# 延長が効かない（実測バグ）。conf の mtime 変化で延ばす。
limit = min(KEEPALIVE_MINUTES, MAX_MINUTES)
conf_mtime = _conf_mtime()
while i < limit:
    i += 1
    RUN, KEEPALIVE_MINUTES = _read_keepalive_conf()   # 都度 ON/OFF + 時間確認
    if not RUN:
        print(f"RUN=0 で {i}周目にキープアライブ終了")
        break
    _m = _conf_mtime()
    if _m != conf_mtime:
        conf_mtime = _m
        limit = min(i - 1 + KEEPALIVE_MINUTES, MAX_MINUTES)
        print(f"♻ アクティビティ検出 → 残り {limit - i + 1} 分に延長"
              f"（合計上限 {MAX_MINUTES} 分）")
    ran_any = True
    time.sleep(60)
    st = _states()
    tunnel_st = kc_proc_state(TUNNEL_NAME, "exact")

    # トンネルが死んだら1回だけ起こす（URL が変わるので取り直して
    # public_url.txt を書き換える。2回目以降は fail 対象）
    # 再起動コマンドと URL 取得はトンネルセルが定義した共通インターフェース。
    if tunnel_st == "DEAD" and tunnel_restarted == 0:
        tunnel_restarted = 1
        try:
            subprocess.run(["pkill", "-x", TUNNEL_NAME], capture_output=True)
            time.sleep(1)
            with open(TUNNEL_LOG, "w") as _lg:
                subprocess.Popen(RESTART_CMD, stdout=_lg, stderr=_lg)
            _new_url = ""
            for _ in range(30):          # 最大30秒で新URL待ち
                time.sleep(1)
                _new_url = probe_url()
                if _new_url:
                    break
            if _new_url:
                url = _new_url
                with open("/tmp/public_url.txt", "w") as _f:
                    _f.write(url + "\n")
                # zip 側も更新しないと、成果物には失効URLが残る
                save_output("public_url.txt", url + "\n")
                print(f"♻ {TUNNEL_NAME} 再起動 → 新URL: {url}")
                # URL 配布（ビーコン）も更新しないと、クライアントは失効URLを掴む
                if "publish_url" in globals():
                    publish_url(url)
                tunnel_st = "alive(restarted)"
                # 新トンネルは別勘定。エッジ伝播待ち中の /healthz 失敗を
                # 旧カウントに混ぜると、稼働中でも次回チェックで偽FAILになる。
                tunnel_dead = 0
                _restarted_now = True
        except Exception as _e:
            print(f"⚠ {TUNNEL_NAME} 再起動失敗: {type(_e).__name__}: {_e}")

    line = (f"[{time.strftime('%H:%M:%S')}] "
            + " ".join(f"{k}={v}" for k, v in st.items())
            + f" {TUNNEL_NAME}={tunnel_st} | GPU: {_gpu()}")

    # pid だけでは tunnel の死は分からない（quick tunnel は URL 失効がありうる）
    # 再起動した同一イテレーションは新URLのエッジ伝播待ちなので判定しない
    if url and i % TUNNEL_CHECK_EVERY == 0 and not _restarted_now:
        try:
            with kc_urlopen_v4(f"{url}/healthz", timeout=15) as r:
                line += " | tunnel=OK" if r.status == 200 else f" | tunnel=HTTP{r.status}"
        except Exception as e:
            tunnel_dead += 1
            line += f" | tunnel=DEAD({type(e).__name__}) x{tunnel_dead}"
            if tunnel_st == "alive":
                line += " ← トンネル生存中だが URL 失効の可能性"
    _restarted_now = False

    log.append(line)
    print(line)

# どれか1つでも死んでいれば失敗扱い（tunnel は pid と併せて判定）。
# tunnel は1回の瞬断で全体を FAIL にしない（稼働後に成果物が死なないように）。
# 2回以上 = 恒久的な失効とみなし、再起動が効かなかった場合もこれで拾う。
# unknown（pgrep 例外/timeout）は確認不能 = 安全側で失敗扱いにする。
# ただし 1 周も回っていない（RUN=0 即時終了）場合は判定材料が無いので fail しない。
failed = False
if not ran_any:
    st = _states()
    tunnel_st = kc_proc_state(TUNNEL_NAME, "exact")
    print(f"ℹ 即時終了のため死活判定はスキップ（{st} {TUNNEL_NAME}={tunnel_st}）")
else:
    failed = (any(v in ("DEAD", "unknown") for v in st.values())
              or tunnel_st in ("DEAD", "unknown")
              or tunnel_dead >= 2)
report = "\n".join([
    f"result={'DEAD' if failed else ('stopped' if not RUN else 'alive')}",
    f"RUN={RUN}",
    f"minutes={KEEPALIVE_MINUTES}",
    f"ran_any={ran_any}",
] + [f"{k}={v}" for k, v in st.items()] + [
    f"tunnel={TUNNEL_NAME}:{tunnel_st}",
    f"url={url}",
    "",
] + log)
# 保存に失敗すると「生きている証跡」が残らないので、失敗も 1 件は証跡を残す
saved = save_output("keepalive.txt", report + "\n")
print()
print("⏱ キープアライブ終了。COMPLETE で /kaggle/working が zip 化されます")

# ============================================================
# 後片付け: GPU を明示的に手放す
# ============================================================
# これをやらないと、モデルが VRAM に固定されたままカーネルが
# COMPLETE になっても GPU を掴み続ける。プロセスを落としても
# Kaggle 側の枠開放は保証されないが、少なくとも VRAM は即座に返る。
print()
print("=" * 56)
print("🧹 後片付け: GPU / プロセスを解放")
print("=" * 56)

# --- 1) エンジン固有の後片付け（モデルのアンロード / キャッシュ退避 など）---
# 順序が効く: プロセスを殺す前にエンジン自身へ VRAM を返させる。
# 未使用のエンジンは settings 側で None を返す（契約上は必須）。
unloaded = engine_cleanup()

# --- 2) プロセスを落とす（SIGTERM → 待機 → SIGKILL）---
# 順序: 外側（トンネル）→ 内側（プロキシ）→ 本体。CLEANUP_TARGETS が持つ。
left = kc_stop_processes(CLEANUP_TARGETS)

# --- 3) 最終確認 ---
print()
if left:
    print(f"  ⚠ 残存: {', '.join(left)}")
else:
    print(f"  ✅ 全プロセス停止（{ENGINE_LABEL} / tunnel / proxy）")

# 後片付けの結果も証跡として残す
save_output("cleanup.txt", "\n".join([
    f"engine_cleanup={unloaded}",
    f"leftover={','.join(left) if left else 'none'}",
    "",
]) + "\n")
print("  📄 cleanup.txt 保存済")
print("=" * 56)

if failed:
    fail("死活確認に失敗しました（keepalive.txt を確認）", ENGINE_LOG)
elif not saved:
    fail("keepalive.txt を保存できませんでした（成果物ディレクトリを確認）")
